# Student Performance Analysis
Run from the `notebooks/` folder after `python generate_dataset.py` in the project root.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
df = pd.read_csv('../data/student_performance.csv')
df.head()

## Data understanding

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print('Duplicates:', df.duplicated().sum())
df.describe().round(2)

## Distributions

In [ ]:
df.hist(bins=25, figsize=(12,8)); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(12,5)); sns.boxplot(data=df.drop(columns=['backlogs'])); plt.show()

## Correlation
Values near +1 or -1 mean a strong relationship with the target.

In [ ]:
plt.figure(figsize=(8,6)); sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0); plt.show()

## Features vs target

In [ ]:
features=[c for c in df.columns if c!='final_score']
fig,axes=plt.subplots(2,3,figsize=(14,8))
for ax,c in zip(axes.ravel(),features):
    sns.scatterplot(data=df,x=c,y='final_score',alpha=.4,ax=ax)
plt.tight_layout(); plt.show()

## Models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X, y = df[features], df['final_score']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
models = {
 'Linear Regression': Pipeline([('s',StandardScaler()),('m',LinearRegression())]),
 'Decision Tree': DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=42),
 'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=42),
}
rows=[]
for n,m in models.items():
    m.fit(X_tr,y_tr); p=m.predict(X_te)
    rows.append([n, mean_absolute_error(y_te,p), np.sqrt(mean_squared_error(y_te,p)), r2_score(y_te,p)])
pd.DataFrame(rows, columns=['Model','MAE','RMSE','R2']).round(3)